# Digital Marketing Campaign Analysis & Revenue Intelligence
### End-to-End Exploratory Data Analysis & Machine Learning Pipeline

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

## 2. Load Dataset

In [ ]:
# Load dataset from current directory or parent directory
data_path = "online_advertising_performance_data.csv" if os.path.exists("online_advertising_performance_data.csv") else "../online_advertising_performance_data.csv"
df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
df.head()

## 3. Data Cleaning & Handling Missing Values

In [ ]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna("Unknown")
    else:
        df[col] = df[col].fillna(0)

print("Missing values handled successfully.")

## 4. Feature Engineering (CTR, CPC, ROI)

In [ ]:
df['CTR'] = df['clicks'] / df['displays'].replace(0, np.nan)
df['CPC'] = df['cost'] / df['clicks'].replace(0, np.nan)
df['ROI'] = (df['revenue'] - df['cost']) / df['cost'].replace(0, np.nan)

df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

df[['displays', 'clicks', 'cost', 'revenue', 'CTR', 'CPC', 'ROI']].head()

## 5. Extended Exploratory Data Analysis (EDA)

In [ ]:
print("Statistical Summary:")
print(df.describe())

# Feature Distributions
df.hist(figsize=(15, 10))
plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.show()

# Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include=['int64', 'float64']).corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

## 6. Feature & Target Separation

In [ ]:
X = df.drop(['revenue'], axis=1)
y = df['revenue']

categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(exclude=['object']).columns

print("Categorical features:", list(categorical_cols))
print("Numerical features:", list(numerical_cols))

## 7. Preprocessing Pipeline (One-Hot Encoding & Feature Scaling)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

## 8. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train size: {X_train.shape[0]} samples, Test size: {X_test.shape[0]} samples")

## 9. Feature Selection

In [ ]:
k_value = min(20, X.shape[1])
feature_selector = SelectKBest(score_func=f_regression, k=k_value)

## 10. Model Definitions

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=300, random_state=42),
    "SVR": SVR()
}

## 11. Model Training & Cross-Validation Evaluation

In [ ]:
results = []
best_model = None
best_score = -np.inf

for name, model in models.items():
    print("=" * 35)
    print(f"Training: {name}")
    print("=" * 35)

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('feature_selection', feature_selector),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    cv_score = cross_val_score(pipeline, X, y, cv=5, scoring='r2').mean()

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}")
    print(f"Cross-Validation R2: {cv_score:.4f}")

    results.append([name, mae, rmse, r2, cv_score])

    if r2 > best_score:
        best_score = r2
        best_model = pipeline

    plt.figure(figsize=(6, 4))
    plt.scatter(y_pred, y_test - y_pred, alpha=0.6)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.title(f"Residual Plot - {name}")
    plt.xlabel("Predicted Revenue")
    plt.ylabel("Residual")
    plt.show()

## 12. Model Benchmark & Comparison

In [ ]:
results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2", "CV_R2"])
print("\nMODEL COMPARISON LEADERBOARD:")
print(results_df.sort_values(by="R2", ascending=False))

plt.figure(figsize=(8, 5))
sns.barplot(x="Model", y="R2", data=results_df)
plt.title("Model Comparison (R2 Score)")
plt.xticks(rotation=45)
plt.show()

## 13. SHAP Feature Explainability

In [ ]:
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

xgb_model = XGBRegressor(n_estimators=300, random_state=42)
xgb_model.fit(X_processed, y)

explainer = shap.Explainer(xgb_model)
shap_values = explainer(X_processed)

shap.summary_plot(shap_values, X_processed, feature_names=feature_names)

## 14. Save Production Pipeline Model

In [ ]:
output_model_path = "best_advertising_model.pkl" if os.path.exists(".") else "../best_advertising_model.pkl"
joblib.dump(best_model, output_model_path)
print(f"Best model saved successfully to {output_model_path}!")